# Módulos estándar · Más fechas

Continuación de *Fechas* con problemas de calendario resueltos con `datetime`, `calendar` y `zoneinfo`: el día del año, *timestamps* Unix, inicio y fin de mes, sumar meses o días laborables, redondear horas, buscar días concretos (domingos, viernes 13, el tercer sábado de un mes), el ciclo sexagenario chino, un calendario de adviento y una cuenta atrás con horario de verano. Ningún ejercicio depende de la fecha real: "ahora" es siempre un instante fijo. Ideas de [PYnative](https://pynative.com/python-exercises-with-solutions/), [machinelearningplus](https://www.machinelearningplus.com/) y [retosdeprogramacion.com](https://retosdeprogramacion.com/ejercicios).

## 1. Día del año

Escribir `dia_del_año(fecha)` restando el 1 de enero y comprobarla con `timetuple().tm_yday` y con el formato `%j` para todos los días de 2024 (bisiesto) y 2026. Mostrar también cuántos días quedan para terminar el año y obtener la fecha a partir del número de día con `strptime`.

In [1]:
import calendar
from datetime import date, datetime, time, timedelta, timezone


def dia_del_año(fecha):
    return (fecha - date(fecha.year, 1, 1)).days + 1


print(f"{'Fecha':<11} {'día':>4} {'%j':>4} {'quedan':>7}")
for f in (date(2026, 1, 1), date(2026, 5, 30), date(2026, 11, 22), date(2028, 12, 31)):
    quedan = (date(f.year, 12, 31) - f).days
    print(f"{f.isoformat():<11} {dia_del_año(f):>4} {f.strftime('%j'):>4} {quedan:>7}")

print("Día 326 de 2026:", datetime.strptime("2026-326", "%Y-%j").date())
for año in (2024, 2026):
    dias = [date(año, 1, 1) + timedelta(days=k) for k in range(366 if calendar.isleap(año) else 365)]
    assert all(dia_del_año(f) == f.timetuple().tm_yday == int(f.strftime("%j")) for f in dias)
    assert all(datetime.strptime(f"{año}-{dia_del_año(f)}", "%Y-%j").date() == f for f in dias)

Fecha        día   %j  quedan
2026-01-01     1  001     364
2026-05-30   150  150     215
2026-11-22   326  326      39
2028-12-31   366  366       0
Día 326 de 2026: 2026-11-22


## 2. Combinar `date` y `time`

Un barco de ejemplo sale todos los días a las horas de una lista y tarda 50 minutos. Construir con `datetime.combine` las salidas del 30 de mayo de 2026 y calcular las llegadas, observando que la última llega al día siguiente. Separar de nuevo la fecha y la hora con `.date()` y `.time()` y crear la misma salida en UTC pasando `tzinfo`.

In [2]:
dia = date(2026, 5, 30)
horario = [time(7, 0), time(11, 30), time(16, 45), time(23, 30)]
travesia = timedelta(minutes=50)

print(f"{'Salida':<17} {'Llegada':<17}")
for hora in horario:
    salida = datetime.combine(dia, hora)
    llegada = salida + travesia
    aviso = "  (al día siguiente)" if llegada.date() != dia else ""
    print(f"{salida:%d/%m/%Y %H:%M}  {llegada:%d/%m/%Y %H:%M}{aviso}")
    assert salida.date() == dia and salida.time() == hora

en_utc = datetime.combine(dia, horario[0], tzinfo=timezone.utc)
print("Con zona:", en_utc.isoformat())
assert en_utc.utcoffset() == timedelta(0) and en_utc.replace(tzinfo=None) == datetime.combine(dia, horario[0])

Salida            Llegada          
30/05/2026 07:00  30/05/2026 07:50
30/05/2026 11:30  30/05/2026 12:20
30/05/2026 16:45  30/05/2026 17:35
30/05/2026 23:30  31/05/2026 00:20  (al día siguiente)
Con zona: 2026-05-30T07:00:00+00:00


## 3. *Timestamps* Unix

Un *timestamp* Unix es el número de segundos desde el 1 de enero de 1970 a las 00:00 UTC. Convertir en los dos sentidos usando siempre `timezone.utc` (un `datetime` sin zona se interpretaría con la hora local del equipo), traducir un *timestamp* en milisegundos y encontrar la fecha del efecto 2038, cuando se desborda un entero con signo de 32 bits. Escribir una conversión que funcione también con *timestamps* negativos.

In [3]:
EPOCH = datetime(1970, 1, 1, tzinfo=timezone.utc)

momento = datetime(2026, 11, 22, 18, 30, tzinfo=timezone.utc)
ts = momento.timestamp()
print(f"{momento.isoformat()} -> {ts:.0f}")
print("1 800 000 000 ->", datetime.fromtimestamp(1_800_000_000, timezone.utc).isoformat())
print("1 700 000 000 000 ms ->", datetime.fromtimestamp(1_700_000_000_000 / 1000, timezone.utc).isoformat())
print("2³¹ - 1 ->", datetime.fromtimestamp(2**31 - 1, timezone.utc).isoformat())
assert datetime.fromtimestamp(ts, timezone.utc) == momento
assert ts == (momento - EPOCH).total_seconds()


def desde_timestamp(segundos):
    # fromtimestamp falla en Windows con valores negativos; sumar a EPOCH funciona en todos los sistemas
    return EPOCH + timedelta(seconds=segundos)


print("-86 400 ->", desde_timestamp(-86_400).isoformat())
assert desde_timestamp(-86_400) == datetime(1969, 12, 31, tzinfo=timezone.utc)
assert all(desde_timestamp(s) == datetime.fromtimestamp(s, timezone.utc) for s in range(0, 2**31, 9_999_991))

2026-11-22T18:30:00+00:00 -> 1795372200
1 800 000 000 -> 2027-01-15T08:00:00+00:00
1 700 000 000 000 ms -> 2023-11-14T22:13:20+00:00
2³¹ - 1 -> 2038-01-19T03:14:07+00:00
-86 400 -> 1969-12-31T00:00:00+00:00


## 4. Inicio y fin de mes, y el próximo lunes

Escribir `primer_dia(fecha)` y `ultimo_dia(fecha)` del mes (con `calendar.monthrange`, que devuelve el día de la semana del día 1 y el número de días) y `proximo(fecha, dia_semana)`, el siguiente día de la semana pedido estrictamente posterior. Comprobarlas con todos los días de 2026 a 2028.

In [4]:
def primer_dia(fecha):
    return fecha.replace(day=1)


def ultimo_dia(fecha):
    return fecha.replace(day=calendar.monthrange(fecha.year, fecha.month)[1])


DIAS = ["lunes", "martes", "miércoles", "jueves", "viernes", "sábado", "domingo"]


def proximo(fecha, dia_semana=calendar.MONDAY):
    return fecha + timedelta(days=(dia_semana - fecha.weekday() - 1) % 7 + 1)


print(f"{'Fecha':<11} {'primero':<11} {'último':<11} {'próx. lunes':<11}")
for f in (date(2026, 11, 22), date(2028, 2, 10), date(2026, 12, 28)):
    print(f"{f.isoformat():<11} {primer_dia(f).isoformat():<11} {ultimo_dia(f).isoformat():<11} {proximo(f).isoformat()}")
print("monthrange(2028, 2):", calendar.monthrange(2028, 2), "-> empieza en", DIAS[calendar.monthrange(2028, 2)[0]])

f = date(2026, 1, 1)
while f.year <= 2028:
    assert (ultimo_dia(f) + timedelta(days=1)).day == 1 and ultimo_dia(f).month == f.month == primer_dia(f).month
    for d in range(7):
        assert proximo(f, d) == next(f + timedelta(days=k) for k in range(1, 8) if (f + timedelta(days=k)).weekday() == d)
    f += timedelta(days=1)

Fecha       primero     último      próx. lunes
2026-11-22  2026-11-01  2026-11-30  2026-11-23
2028-02-10  2028-02-01  2028-02-29  2028-02-14
2026-12-28  2026-12-01  2026-12-31  2027-01-04
monthrange(2028, 2): (calendar.TUESDAY, 29) -> empieza en martes


## 5. Sumar meses

Escribir `sumar_meses(fecha, n)` sin librerías externas (`n` puede ser negativo): si el día no existe en el mes de destino, se usa su último día. Ver que sumar un mes dos veces no siempre equivale a sumar dos meses y comprobar las reglas con todas las fechas de 2027 a 2028 y desplazamientos de −24 a 24 meses.

In [5]:
def sumar_meses(fecha, n):
    año, mes = divmod(fecha.year * 12 + fecha.month - 1 + n, 12)
    mes += 1
    return fecha.replace(year=año, month=mes, day=min(fecha.day, calendar.monthrange(año, mes)[1]))


casos = [(date(2026, 1, 31), 1), (date(2028, 1, 31), 1), (date(2026, 8, 31), 1), (date(2026, 11, 15), 3),
         (date(2026, 3, 31), -1), (date(2028, 2, 29), 12)]
for f, n in casos:
    print(f"{f.isoformat()} {n:+3d} {'mes' if abs(n) == 1 else 'meses':<5} -> {sumar_meses(f, n).isoformat()}")

uno_y_uno = sumar_meses(sumar_meses(date(2026, 1, 31), 1), 1)
print(f"31/01 + 1 + 1 = {uno_y_uno.isoformat()}, pero 31/01 + 2 = {sumar_meses(date(2026, 1, 31), 2).isoformat()}")

f = date(2027, 1, 1)
while f.year <= 2028:
    for n in range(-24, 25):
        r = sumar_meses(f, n)
        assert (r.year * 12 + r.month) - (f.year * 12 + f.month) == n
        assert r.day == f.day or (r.day < f.day and r == ultimo_dia(r))
    f += timedelta(days=1)

2026-01-31  +1 mes   -> 2026-02-28
2028-01-31  +1 mes   -> 2028-02-29
2026-08-31  +1 mes   -> 2026-09-30
2026-11-15  +3 meses -> 2027-02-15
2026-03-31  -1 mes   -> 2026-02-28
2028-02-29 +12 meses -> 2029-02-28
31/01 + 1 + 1 = 2026-03-28, pero 31/01 + 2 = 2026-03-31


## 6. Redondear un `datetime`

Escribir `redondear(momento, paso)`, que lleve un `datetime` al múltiplo de `paso` más cercano (por defecto, la hora en punto; los empates hacia arriba), usando que `timedelta % timedelta` da el resto. Probarlo con medias horas, con un cambio de día y con pasos de 15 minutos, y comprobarlo con 1000 instantes aleatorios contra los dos candidatos (abajo y arriba).

In [6]:
import random


def redondear(momento, paso=timedelta(hours=1)):
    resto = (momento - datetime.min) % paso
    return momento - resto + (paso if resto >= paso / 2 else timedelta(0))


pruebas = [(datetime(2026, 11, 22, 10, 29, 59), timedelta(hours=1)),
           (datetime(2026, 11, 22, 10, 30), timedelta(hours=1)),
           (datetime(2026, 12, 31, 23, 45), timedelta(hours=1)),
           (datetime(2026, 11, 22, 10, 7, 30), timedelta(minutes=15)),
           (datetime(2026, 11, 22, 10, 7, 29), timedelta(minutes=15))]
for momento, paso in pruebas:
    print(f"{momento:%d/%m/%Y %H:%M:%S} a {paso.seconds // 60:>2} min -> {redondear(momento, paso):%d/%m/%Y %H:%M}")

rng = random.Random(2026)
for _ in range(1_000):
    momento = datetime(2026, 1, 1) + timedelta(seconds=rng.randrange(365 * 86_400))
    paso = timedelta(minutes=rng.choice([5, 15, 30, 60]))
    abajo = momento.replace(minute=0, second=0) + paso * ((momento.minute * 60 + momento.second) // paso.seconds)
    esperado = min(abajo, abajo + paso, key=lambda c: (abs(c - momento), c == abajo))
    assert redondear(momento, paso) == esperado

22/11/2026 10:29:59 a 60 min -> 22/11/2026 10:00
22/11/2026 10:30:00 a 60 min -> 22/11/2026 11:00
31/12/2026 23:45:00 a 60 min -> 01/01/2027 00:00
22/11/2026 10:07:30 a 15 min -> 22/11/2026 10:15
22/11/2026 10:07:29 a 15 min -> 22/11/2026 10:00


## 7. Todos los domingos de un año

Escribir el generador `domingos(año)`, que empiece en el primer domingo y avance de semana en semana, y comprobarlo filtrando todos los días del año. Mostrar cuántos domingos tiene 2026 y qué años entre 2020 y 2040 tienen 53, comprobando la regla: el 1 de enero cae en domingo, o en sábado si el año es bisiesto.

In [7]:
def domingos(año):
    f = date(año, 1, 1)
    f += timedelta(days=(calendar.SUNDAY - f.weekday()) % 7)
    while f.year == año:
        yield f
        f += timedelta(weeks=1)


lista = list(domingos(2026))
print(f"2026 tiene {len(lista)} domingos: {lista[0]:%d/%m}, {lista[1]:%d/%m}, … , {lista[-1]:%d/%m}")
print("Con 53 domingos:", [a for a in range(2020, 2041) if len(list(domingos(a))) == 53])

for año in range(1990, 2061):
    todos = [date(año, 1, 1) + timedelta(days=k) for k in range(366 if calendar.isleap(año) else 365)]
    assert list(domingos(año)) == [f for f in todos if f.weekday() == calendar.SUNDAY]
    dia1 = date(año, 1, 1).weekday()
    regla = dia1 == calendar.SUNDAY or (calendar.isleap(año) and dia1 == calendar.SATURDAY)
    assert (len(list(domingos(año))) == 53) == regla

2026 tiene 52 domingos: 04/01, 11/01, … , 27/12
Con 53 domingos: [2023, 2028, 2034, 2040]


## 8. Sumar días laborables

Escribir `sumar_laborables(fecha, n, festivos)`, que avance (o retroceda, si `n` es negativo) `n` días laborables saltando sábados, domingos y los festivos de la lista. Probarlo con unos festivos de ejemplo de 2026 y comprobarlo con `numpy.busday_offset` en 2000 casos aleatorios. Con `n = 0` se devuelve la misma fecha.

In [8]:
import numpy as np

FESTIVOS_2026 = [date(2026, 1, 1), date(2026, 1, 6), date(2026, 4, 2), date(2026, 4, 3), date(2026, 5, 1),
                 date(2026, 10, 12), date(2026, 12, 8), date(2026, 12, 25)]


def sumar_laborables(fecha, n, festivos=()):
    festivos, paso = set(festivos), 1 if n >= 0 else -1
    for _ in range(abs(n)):
        fecha += timedelta(days=paso)
        while fecha.weekday() >= 5 or fecha in festivos:
            fecha += timedelta(days=paso)
    return fecha


for f, n in ((date(2026, 4, 1), 3), (date(2026, 11, 20), 1), (date(2026, 12, 7), 5), (date(2026, 1, 7), -2)):
    sin, con = sumar_laborables(f, n), sumar_laborables(f, n, FESTIVOS_2026)
    print(f"{DIAS[f.weekday()]:>9} {f:%d/%m} {n:+d} -> sin festivos {DIAS[sin.weekday()]} {sin:%d/%m}; "
          f"con festivos {DIAS[con.weekday()]} {con:%d/%m}")

rng = random.Random(2026)
for _ in range(2_000):
    f = date(2026, 1, 1) + timedelta(days=rng.randrange(365))
    n = rng.choice([k for k in range(-15, 16) if k])  # con n = 0 numpy movería un sábado al viernes
    # Si se parte de un día no laborable, numpy primero lo mueve: hacia atrás al avanzar y hacia delante al retroceder
    roll = "backward" if n >= 0 else "forward"
    esperado = np.busday_offset(f, n, roll=roll, holidays=FESTIVOS_2026).astype(date)
    assert sumar_laborables(f, n, FESTIVOS_2026) == esperado
assert sumar_laborables(date(2026, 11, 21), 0) == date(2026, 11, 21)

miércoles 01/04 +3 -> sin festivos lunes 06/04; con festivos miércoles 08/04


  viernes 20/11 +1 -> sin festivos lunes 23/11; con festivos lunes 23/11
    lunes 07/12 +5 -> sin festivos lunes 14/12; con festivos martes 15/12
miércoles 07/01 -2 -> sin festivos lunes 05/01; con festivos viernes 02/01


## 9. Viernes 13

Escribir `tiene_viernes_13(año, mes)` y listar los viernes 13 de la década 2020-2029 agrupados por año. Comprobar que todos los años de 1900 a 2100 tienen al menos uno y como mucho tres.

In [9]:
def tiene_viernes_13(año, mes):
    return date(año, mes, 13).weekday() == calendar.FRIDAY


MESES = ["enero", "febrero", "marzo", "abril", "mayo", "junio", "julio", "agosto", "septiembre", "octubre",
         "noviembre", "diciembre"]
for año in range(2020, 2030):
    meses = [MESES[m - 1] for m in range(1, 13) if tiene_viernes_13(año, m)]
    print(f"{año}: {', '.join(meses)}")

decada = [date(a, m, 13) for a in range(2020, 2030) for m in range(1, 13) if tiene_viernes_13(a, m)]
print("Total en la década:", len(decada))
inicio, fin = date(2020, 1, 1), date(2030, 1, 1)
fuerza_bruta = [inicio + timedelta(days=k) for k in range((fin - inicio).days)]
assert decada == [f for f in fuerza_bruta if f.day == 13 and f.weekday() == calendar.FRIDAY]
assert all(1 <= sum(tiene_viernes_13(a, m) for m in range(1, 13)) <= 3 for a in range(1900, 2101))

2020: marzo, noviembre
2021: agosto
2022: mayo
2023: enero, octubre
2024: septiembre, diciembre
2025: junio
2026: febrero, marzo, noviembre
2027: agosto
2028: octubre
2029: abril, julio
Total en la década: 16


## 10. Ciclo sexagenario chino

En el calendario chino los años se nombran combinando 10 "troncos" (cinco elementos, dos años seguidos cada uno: madera, fuego, tierra, metal y agua) y 12 animales; como $\text{mcm}(10, 12) = 60$, la combinación se repite cada 60 años. El ciclo empezó un año de la rata de madera, por ejemplo 1984. Escribir `año_chino(año)` y comprobarla con años conocidos. El año chino empieza entre el 21 de enero y el 20 de febrero, así que esto vale para la mayor parte del año gregoriano, no para sus primeras semanas.

In [10]:
import math

ELEMENTOS = ["madera", "fuego", "tierra", "metal", "agua"]
ANIMALES = ["rata", "buey", "tigre", "conejo", "dragón", "serpiente", "caballo", "cabra", "mono", "gallo", "perro",
            "cerdo"]


def año_chino(año):
    k = año - 1984
    return ANIMALES[k % 12], ELEMENTOS[k % 10 // 2], k % 60 + 1


for año in (1984, 2000, 2020, 2024, 2025, 2026, 2027, 2044):
    animal, elemento, posicion = año_chino(año)
    print(f"{año}: {animal} de {elemento} (año {posicion} del ciclo)")

conocidos = {1984: ("rata", "madera"), 2000: ("dragón", "metal"), 2020: ("rata", "metal"),
             2024: ("dragón", "madera"), 2025: ("serpiente", "madera"), 2026: ("caballo", "fuego")}
assert all(año_chino(a)[:2] == esperado for a, esperado in conocidos.items())
assert math.lcm(10, 12) == 60 and all(año_chino(a)[:2] == año_chino(a + 60)[:2] for a in range(1900, 2100))
assert len({año_chino(a)[:2] for a in range(1984, 2044)}) == 60

1984: rata de madera (año 1 del ciclo)
2000: dragón de metal (año 17 del ciclo)
2020: rata de metal (año 37 del ciclo)
2024: dragón de madera (año 41 del ciclo)
2025: serpiente de madera (año 42 del ciclo)
2026: caballo de fuego (año 43 del ciclo)
2027: cabra de fuego (año 44 del ciclo)
2044: rata de madera (año 1 del ciclo)


## 11. Calendario de adviento

Un calendario de adviento tiene un regalo para cada día del 1 al 24 de diciembre. Escribir `adviento(momento)`, que diga si el calendario aún no ha empezado (y cuánto falta), qué regalo toca y cuánto queda hasta el 24 de diciembre, o que ya terminó. Escribir también `duracion_legible(td)`, que muestre un `timedelta` en días, horas y minutos.

In [11]:
REGALOS = ["una postal del Teide", "un paquete de gofio", "un frasco de mojo verde", "unas truchas de batata",
           "unos calcetines de lana", "un timple de juguete", "una taza", "un puzle de 500 piezas", "un libro de cuentos",
           "una bufanda", "unas castañas asadas", "un llavero", "unos bombones", "una vela", "un cuaderno",
           "una planta suculenta", "un queso tierno", "unos lápices de colores", "una camiseta", "un marcapáginas",
           "una baraja española", "un bizcocho", "unas almendras", "un abrazo"]


def duracion_legible(td):
    horas, resto = divmod(td.seconds, 3600)
    return f"{td.days} días, {horas} h y {resto // 60} min"


def adviento(momento):
    inicio, nochebuena = datetime(momento.year, 12, 1), datetime(momento.year, 12, 24)
    if momento < inicio:
        return f"Aún no ha empezado: faltan {duracion_legible(inicio - momento)}"
    if momento.date() > nochebuena.date():
        return f"El calendario de {momento.year} ya terminó"
    regalo = f"Día {momento.day}: {REGALOS[momento.day - 1]}"
    if momento.day == 24:
        return regalo + ". ¡Hoy es Nochebuena!"
    return f"{regalo}. Para el 24 faltan {duracion_legible(nochebuena - momento)}"


for momento in (datetime(2026, 11, 22, 18, 30), datetime(2026, 12, 1, 9, 0), datetime(2026, 12, 13, 20, 15),
                datetime(2026, 12, 24, 10, 0), datetime(2026, 12, 27, 12, 0)):
    print(f"{momento:%d/%m %H:%M}  {adviento(momento)}")

assert len(REGALOS) == 24 and len(set(REGALOS)) == 24
for dia in range(1, 24):
    texto = adviento(datetime(2026, 12, dia))
    assert REGALOS[dia - 1] in texto and f"faltan {24 - dia} días, 0 h y 0 min" in texto
assert duracion_legible(timedelta(days=2, hours=5, minutes=7, seconds=59)) == "2 días, 5 h y 7 min"

22/11 18:30  Aún no ha empezado: faltan 8 días, 5 h y 30 min
01/12 09:00  Día 1: una postal del Teide. Para el 24 faltan 22 días, 15 h y 0 min
13/12 20:15  Día 13: unos bombones. Para el 24 faltan 10 días, 3 h y 45 min
24/12 10:00  Día 24: un abrazo. ¡Hoy es Nochebuena!
27/12 12:00  El calendario de 2026 ya terminó


## 12. Cuenta atrás con `zoneinfo`

Tomando como "ahora" el 22 de noviembre de 2026 a las 18:30 en Canarias, calcular cuánto falta para la medianoche de Año Nuevo y para el mediodía del 30 de mayo de 2027 en la zona `Atlantic/Canary`, convirtiendo cada fecha a UTC. Comprobar que restar dos `datetime` con la misma zona da la diferencia de reloj y no el tiempo real (en el segundo caso, el cambio al horario de verano quita una hora), y reproducir las conversiones a mano con `timezone(timedelta(hours=...))`.

In [12]:
from zoneinfo import ZoneInfo

# En Windows no hay base de datos de zonas horarias del sistema: zoneinfo usa el paquete tzdata
CANARIAS = ZoneInfo("Atlantic/Canary")
ahora = datetime(2026, 11, 22, 18, 30, tzinfo=CANARIAS)
eventos = {"Año Nuevo": datetime(2027, 1, 1, 0, 0, tzinfo=CANARIAS),
           "30 de mayo, 12:00": datetime(2027, 5, 30, 12, 0, tzinfo=CANARIAS)}

print(f"Ahora: {ahora:%d/%m/%Y %H:%M} {ahora.tzname()} = {ahora.astimezone(timezone.utc):%H:%M} UTC")
for nombre, momento in eventos.items():
    en_utc = momento.astimezone(timezone.utc)
    real = en_utc - ahora.astimezone(timezone.utc)
    de_reloj = momento - ahora
    print(f"{nombre:<18} {momento:%d/%m/%Y %H:%M} {momento.tzname()} = {en_utc:%d/%m/%Y %H:%M} UTC; "
          f"faltan {duracion_legible(real)} (restando sin pasar a UTC: {duracion_legible(de_reloj)})")
    assert de_reloj - real == momento.utcoffset() - ahora.utcoffset()

assert eventos["30 de mayo, 12:00"].utcoffset() == timedelta(hours=1) and ahora.utcoffset() == timedelta(0)

Ahora: 22/11/2026 18:30 WET = 18:30 UTC
Año Nuevo          01/01/2027 00:00 WET = 01/01/2027 00:00 UTC; faltan 39 días, 5 h y 30 min (restando sin pasar a UTC: 39 días, 5 h y 30 min)
30 de mayo, 12:00  30/05/2027 12:00 WEST = 30/05/2027 11:00 UTC; faltan 188 días, 16 h y 30 min (restando sin pasar a UTC: 188 días, 17 h y 30 min)


Sin `zoneinfo` hay que conocer las reglas: Canarias usa UTC+0 (WET) en invierno y UTC+1 (WEST) desde el último domingo de marzo hasta el último domingo de octubre. Con zonas fijas, la conversión es la misma.

In [13]:
WET, WEST = timezone(timedelta(0), "WET"), timezone(timedelta(hours=1), "WEST")
a_mano = {"Año Nuevo": datetime(2027, 1, 1, 0, 0, tzinfo=WET),
          "30 de mayo, 12:00": datetime(2027, 5, 30, 12, 0, tzinfo=WEST)}
for nombre, momento in a_mano.items():
    print(f"{nombre:<18} {momento.isoformat()} = {momento.astimezone(timezone.utc).isoformat()}")
    assert momento == eventos[nombre] and momento.tzname() == eventos[nombre].tzname()

Año Nuevo          2027-01-01T00:00:00+00:00 = 2027-01-01T00:00:00+00:00
30 de mayo, 12:00  2027-05-30T12:00:00+01:00 = 2027-05-30T11:00:00+00:00


## 13. El enésimo día de la semana de un mes

Escribir `enesimo(año, mes, dia_semana, n)`, que devuelva por ejemplo el tercer sábado de un mes calculando el desplazamiento desde el día 1 (y lance `ValueError` si el mes no tiene tantos). Mostrar el tercer sábado de septiembre de 2026 a 2035 y el cuarto jueves de noviembre de 2026, y comprobarla contra `calendar.monthcalendar`.

In [14]:
def enesimo(año, mes, dia_semana, n):
    primero = date(año, mes, 1)
    fecha = primero + timedelta(days=(dia_semana - primero.weekday()) % 7 + 7 * (n - 1))
    if fecha.month != mes:
        raise ValueError(f"{MESES[mes - 1]} de {año} no tiene {n} {DIAS[dia_semana]}")
    return fecha


print("Tercer sábado de septiembre:")
for año in range(2026, 2036):
    print(f"  {año}: {enesimo(año, 9, calendar.SATURDAY, 3):%d/%m}")
print("Cuarto jueves de noviembre de 2026:", enesimo(2026, 11, calendar.THURSDAY, 4).isoformat())
try:
    enesimo(2026, 2, calendar.MONDAY, 5)
except ValueError as e:
    print("Error:", e)

for año in range(2000, 2051):
    for mes in range(1, 13):
        for d in range(7):
            dias = [semana[d] for semana in calendar.monthcalendar(año, mes) if semana[d]]
            assert [enesimo(año, mes, d, n).day for n in range(1, len(dias) + 1)] == dias

Tercer sábado de septiembre:
  2026: 19/09
  2027: 18/09
  2028: 16/09
  2029: 15/09
  2030: 21/09
  2031: 20/09
  2032: 18/09
  2033: 17/09
  2034: 16/09
  2035: 15/09
Cuarto jueves de noviembre de 2026: 2026-11-26
Error: febrero de 2026 no tiene 5 lunes
